In [0]:
"""
config.py
---------
Central configuration for the Carousell S3 -> S3 medallion pipeline.
Update BUCKET and the *_PREFIX values to match your environment.
All paths are S3 URIs so the same code runs on Databricks (with the
S3 bucket mounted / IAM instance profile attached) or via spark-submit
with the hadoop-aws + aws-java-sdk-bundle packages on the cluster.
"""

# ---------------------------------------------------------------------------
# S3 layout
# ---------------------------------------------------------------------------
BUCKET = "s3://destiny-corousell-target"

# RAW_PREFIX     = f"{BUCKET}/raw/orders"          # original source CSVs land here (MySQL/SQLServer/API dumps)
RAW_PREFIX = f"{BUCKET}/raw"
BRONZE_PREFIX  = f"{BUCKET}/Raw data bronze"       # raw data copied 1:1 into Delta, ingestion metadata added
SILVER_PREFIX  = f"{BUCKET}/clean_silver"       # cleaned, standardized, deduplicated, FK-validated
QUARANTINE_PREFIX = f"{BUCKET}/quarantine"  # rows that fail validation, kept for DQ reporting
GOLD_PREFIX    = f"{BUCKET}/gold"         # star-schema dims + facts, ready for Power BI
DQ_METRICS_PATH = f"{BUCKET}/dq_metrics"  # one row per pipeline run per table with pass/fail counts


# ---------------------------------------------------------------------------
# Table -> raw file name mapping (source system CSVs, see project brief)
# ---------------------------------------------------------------------------
RAW_FILES = {
    "users": "users.csv",
    "sellers": "sellers.csv",
    "products": "products.csv",
    "orders": "orders.csv",
    "payments": "payments.csv",
    "shipping": "shipping.csv",
    "reviews": "reviews.csv",
    "wishlist": "wishlist.csv",
    "product_images": "product_images.csv",
    "country_master": "country_master.csv",
    "category_master": "category_master.csv",
    "exchange_rates": "exchange_rates.csv",
}

# ---------------------------------------------------------------------------
# Reference / lookup constants used across cleaning scripts
# ---------------------------------------------------------------------------
VALID_ORDER_STATUSES = ["Completed", "Pending", "Cancelled", "Refunded", "Processing", "Shipped"]
VALID_COUNTRY_CODES = ["SG", "MY", "ID", "HK", "TW", "AU", "PH"]
VALID_CURRENCIES = ["SGD", "MYR", "IDR", "HKD", "TWD", "AUD", "PHP"]

ORDER_STATUS_MAP = {
    "complet": "Completed", "completed": "Completed",
    "cancelld": "Cancelled", "cancelled": "Cancelled",
    "in_progress": "Processing", "processing": "Processing",
    "shipped??": "Shipped", "shipped": "Shipped",
    "pending": "Pending", "refunded": "Refunded",
}

VALID_PAYMENT_STATUSES = ["Success", "Pending", "Failed", "Refunded", "Chargeback"]
PAYMENT_STATUS_MAP = {
    "succes": "Success", "success": "Success",
    "faild": "Failed", "failed": "Failed",
    "in_review": "Pending", "pending": "Pending",
    "refunded": "Refunded", "chargeback": "Chargeback",
}

VALID_SHIPPING_STATUSES = ["Delivered", "In Transit", "Pending Pickup", "Returned", "Lost", "Cancelled"]
SHIPPING_STATUS_MAP = {
    "deliverd": "Delivered", "delivered": "Delivered",
    "in-transit!!": "In Transit", "in transit": "In Transit",
    "shipped??": "In Transit",
    "unknown": "Pending Pickup",
    "cancelled": "Cancelled", "returned": "Returned", "lost": "Lost",
}

NULL_TOKENS = ["", "NULL", "null", "N/A", "n/a", "NA", "UNKNOWN", "unknown", None]

# Broken/synthetic FK ids are injected in the 900000-999999 range in this dataset;
# treat anything in that range (or not present in the parent dim) as broken.
BROKEN_FK_RANGE = (900000, 999999)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import when, col

### admissions data ingestions
# Define S3 path
def path_s3_bucket():
    data_source = "admissions.csv"
    bucket = "source-system-mysql-sqlservers"
    prefix = f"{data_source}"
    buck = "regis-healthcare/admissions"
    s3_path = f"s3://{bucket}/{buck}/{prefix}"
    return s3_path,data_source
s3_path,data_source = path_s3_bucket()
print(s3_path)
# s3_path = "s3_path"

class data_ingestion:
    def letest_file_update_only(self,s3_path):
        self.s3_path = s3_path
        # print(s3_path)
        # Use dbutils to list files and find the latest
        files = dbutils.fs.ls(s3_path)
        latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]
        return latest_file.path
s1 = data_ingestion()
base_path = s1.letest_file_update_only(s3_path)
print(base_path)
from datetime import datetime
from pyspark.sql.functions import lit
### Read CSV
class read_records:
    def reading_data(self):
        df = (
            spark.read.format("csv")
             .option("header",True)
             .option("inferSchema",True)
            .load(base_path)
            )
        df = df.withColumn("load_ts",
                          F.current_date())
        # self.df = df
        # return df
        return df

asd = read_records()
asd.reading_data()
df = asd.reading_data()
# display(df)

from pyspark.sql.functions import year, month, dayofmonth
# Add year, month, and day columns derived from load_ts
df = df.withColumn("year", year("load_ts")) \
                   .withColumn("month", month("load_ts")) \
                   .withColumn("day", dayofmonth("load_ts"))
display(df)

#Save data
## bronze write to s3
from pyspark.sql.utils import AnalysisException
import traceback

class FinalLoad:
    def loading_final_bronze_order(self, df, data_source):
        try:
            # Attempt to write DataFrame to csv
            df.write.format("csv") \
                .option("header", "true")\
                .mode("overwrite") \
                .partitionBy("year","month","day") \
                .save(f"s3://destiny-regishealthcare-target/raw/{data_source}/")
             # .option("overwriteSchema", "true") \
            print("✅ Data successfully written to RAW layer.")
        
        except AnalysisException as ae:
            print("AnalysisException occurred while writing data:")
            print(ae)
        
        except Exception as e:
            print("Unexpected error occurred while writing data:")
            print(str(e))
            traceback.print_exc()   # optional: prints full stack trace for debugging

# Usage
TAD = FinalLoad()
TAD.loading_final_bronze_order(df, data_source="admissions")